In [21]:
# correct working directory only once 
if not "working_directory_corrected" in vars():
    %cd ..
    working_directory_corrected = True

In [ ]:
import os
from sklearn.model_selection import train_test_split
import numpy as np
from utils.true_distance.generate_true_distance_dataset import generate_true_distance_dataset
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
import pandas as pd 

### Dataset with 1000 and 10000 samples

In this section we create 2 datasets; one with 1000 samples and another one with 10000 samples. We save both locally in our /data folder as a pickle file. 

In [23]:
SAMPLE_SIZES = [1000, 10000]
datasets = {}

for n in SAMPLE_SIZES:
    path = f"data/true_distance_heuristic_dataset_{n}.pkl"
    if os.path.exists(path):
        datasets[n] = pd.read_pickle(path)
    else:
        datasets[n] = generate_true_distance_dataset(num_samples=n, seed=n)
        datasets[n].to_pickle(path)

Initializing Environment and Model...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 2004.93it/s]

11:42:24 | INFO  | utils.py    | Loading checkpoint from folder /Users/tomseidel/.cache/huggingface/hub/models--quentinll--lewm-tworooms/snapshots/77adaae0bc31deab21c93740d1f8bb947cd0bdec...
11:42:24 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
Generating 10000 samples...


Progress: 100/10000
Progress: 200/10000
Progress: 300/10000
Progress: 400/10000
Progress: 500/10000
Progress: 600/10000
Progress: 700/10000
Progress: 800/10000
Progress: 900/10000
Progress: 1000/10000
Progress: 1100/10000
Progress: 1200/10000
Progress: 1300/10000
Progress: 1400/10000
Progress: 1500/10000
Progress: 1600/10000
Progress: 1700/10000
Progress: 1800/10000
Progress: 1900/10000
Progress: 2000/10000
Progress: 2100/10000
Progress: 2200/10000
Progress: 2300/10000
Progress: 2400/10000
Progress: 2500/10000
Progress: 2600/10000
Progress: 2700/10000
Progress: 2800/10000
Progress: 2900/10000
Progress: 3000/10000
Progress: 3100/10000
Progress: 3200/10000
Progress: 3300/10000
Progress: 3400/10000
Progress: 3500/10000
Progress: 3600/10000
Progress: 3700/10000
Progress: 3800/10000
Progress: 3900/10000
Progress: 4000/10000
Progress: 4100/10000
Progress: 4200/10000
Progress: 4300/10000
Progress: 4400/10000
Progress: 4500/10000
Progress: 4600/10000
Progress: 4700/10000
Progress: 4800/10000
P

### Model training

80/20 train/test split with fixed model parameters 

In [ ]:
def heuristic_dataset_to_xy(df):
    agent_encs = np.array(df['agent_pos_enc'].tolist(), dtype=float)
    goal_encs = np.array(df['goal_pos_enc'].tolist(), dtype=float)
    X = np.hstack([agent_encs, goal_encs])
    y = df['heuristic'].values.astype(float)
    return X, y

def train_and_evaluate(X, y, test_size=0.2, random_state=42):
    x_train, x_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "MLP Neural Net": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
    }

    results = {}
    for name, model in models.items():
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
        results[name] = {
            "MAE": mean_absolute_error(y_test, y_pred),
            "MSE": mean_squared_error(y_test, y_pred),
            "R2": r2_score(y_test, y_pred),
            "Pearson": pearsonr(y_test, y_pred)[0],
            "Spearman": spearmanr(y_test, y_pred)[0],
        }

    return pd.DataFrame(results).T

In [25]:
results = {}

for n, df in datasets.items():
    X, y = heuristic_dataset_to_xy(df)
    results[n] = train_and_evaluate(X, y)

### Comparison: 1000 vs 10000 samples

In this section we want to compare our evaluation matrics for all models trained on 1000 vs 10000 samples. This should give us a first clue if we should increase the set size

In [26]:
comparison = pd.concat({f"{n} samples": r for n, r in results.items()})
comparison

MAE           MSE        R2   Pearson  \
1000 samples  Linear Regression  96.159764  23003.355819 -3.586360  0.108705   
              Random Forest      58.476334   5021.872705 -0.001250  0.144834   
              MLP Neural Net     59.144309   5014.515877  0.000217  0.203953   
10000 samples Linear Regression  56.930704   4547.162349  0.126580  0.363024   
              Random Forest      56.832543   4670.717433  0.102847  0.320741   
              MLP Neural Net     60.350915   5157.053353  0.009432  0.110057   

                                 Spearman  
1000 samples  Linear Regression  0.122357  
              Random Forest      0.087871  
              MLP Neural Net     0.168870  
10000 samples Linear Regression  0.343720  
              Random Forest      0.290894  
              MLP Neural Net     0.072965

Linear Regression and Random Forest clearly benefit from more data (R2 and correlations roughly triple going from 1000 to 10000 samples). MLP barely improves and even loses a bit of rank correlation